# Comparison

In [65]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont
from ultralytics import YOLO
from collections import Counter

# Paths
annotation_path = r"C:\Users\Spacelab3\Desktop\envs\InstanceSegmentation\test\labels"                   # Annotation folder
image_path = r"C:\Users\Spacelab3\Desktop\envs\InstanceSegmentation\test\images"                        # Images folder
output_path = r"C:\Users\Spacelab3\Desktop\envs\Segmentation\labeled2"                                  # Output folder
model_path = r"C:\Users\Spacelab3\Desktop\envs\Segmentation\Object Segmentation.v3i.yolov8\best-150.pt"

# Load YOLO model
model = YOLO(model_path)

# Color map for visualization
color_map = {
    0: ((120, 150, 170, 100), (0, 0, 0, 255)),  # Darker Light Blue
    1: ((128, 0, 128, 100), (128, 0, 128, 255)),  # Purple
    2: ((255, 0, 0, 100), (255, 0, 0, 255)),  # Red
    3: ((0, 255, 0, 100), (0, 255, 0, 255)),  # Green
    4: ((255, 20, 147, 100), (255, 20, 147, 255)),  # Medium Violet Red
}
labels = {0: "Car", 1: "Motorcycle", 2: "Person", 3: "Truck", 4: "Van"}

# Function to read annotation file
def read_annotation_file(file_path):
    polygons = []
    with open(file_path, 'r') as file:
        for line in file:
            values = line.strip().split()
            class_id = int(values[0])
            coords = list(map(float, values[1:]))
            polygons.append((class_id, coords))
    return polygons

# Function to convert normalized coordinates to pixel values
def get_coordinates(coords, img_width, img_height):
    scaled_coords = [
        (int(coords[i] * img_width), int(coords[i + 1] * img_height))
        for i in range(0, len(coords), 2)
    ]
    return scaled_coords, min(x for x, _ in scaled_coords), min(y for _, y in scaled_coords)

# Function to process annotated images
def process_annotated_images(image_path, annotation_path):
    image = cv2.imread(image_path, cv2.COLOR_BGR2RGB)
    img_width, img_height = image.shape[1], image.shape[0]
    annotations = read_annotation_file(annotation_path)
    
    img = Image.open(image_path).convert("RGBA")
    overlay = Image.new("RGBA", img.size, (255, 255, 255, 0))  # Transparent overlay
    draw_overlay = ImageDraw.Draw(overlay)
    
    for class_id, coords in annotations:
        pixel_coords, min_x, min_y = get_coordinates(coords, img_width, img_height)
        fill_color, outline_color = color_map.get(class_id, ((0, 0, 0, 100), (0, 0, 0, 255)))
        draw_overlay.polygon(pixel_coords, outline=outline_color, fill=fill_color)
        text_position = (min_x, min_y - 20)
        draw_overlay.text(text_position, f"{labels[class_id]}", fill=outline_color)
    
    return Image.alpha_composite(img, overlay)  

# Function to process YOLO predictions
def process_predictions(image_path):
    image = cv2.imread(image_path, cv2.COLOR_BGR2RGB)
    results = model.predict(source=image)
    
    img = Image.open(image_path).convert("RGBA")
    overlay = Image.new("RGBA", img.size, (255, 255, 255, 0))
    draw_overlay = ImageDraw.Draw(overlay)
    
    if results[0].masks:
        for class_id, i in zip(results[0].boxes.cls, range(len(results[0].masks.xy))):
            x_min, y_min, x_max, y_max = map(int, results[0].boxes.xyxy[i])
            label = labels[int(class_id)]
            fill_color, outline_color = color_map.get(int(class_id), ((0, 0, 0, 100), (0, 0, 0, 255)))

            mask_polygon = results[0].masks.xy[i]
            if mask_polygon.any():
                draw_overlay.polygon(mask_polygon, outline=outline_color, fill=fill_color)
            
            draw_overlay.rectangle([x_min, y_min, x_max, y_max], outline=outline_color, width=2)
            text_position = (x_min, y_min - 20)
            font = ImageFont.truetype("arial.ttf", 20)
            draw_overlay.text(text_position, label, font=font, fill=outline_color)
    
    return Image.alpha_composite(img, overlay)

# Compare YOLO predictions with annotated images
for filename in os.listdir(image_path):
    if filename.endswith(('.jpg', '.jpeg', '.png')):
        img_path = os.path.join(image_path, filename)
        ann_path = os.path.join(annotation_path, f"{filename[:-4]}.txt")
        
        annotated_image = None
        if os.path.exists(ann_path):  # If annotation exists
            annotated_image = process_annotated_images(img_path, ann_path)
        
        yolo_image = process_predictions(img_path)  # YOLO prediction
        
        # Plot side-by-side
        fig, axes = plt.subplots(1, 2, figsize=(12, 6))
        
        if annotated_image:
            axes[0].imshow(annotated_image)
            axes[0].set_title(f"Annotated:")
        else:
            axes[0].set_title("No Annotations Found")
            axes[0].axis("off")
        
        axes[1].imshow(yolo_image)
        axes[1].set_title(f"YOLO Prediction:")
        
        for ax in axes:
            ax.axis("off")
        
        plt.tight_layout()
        plt.savefig(f"{output_path}/{filename}_labeled.jpg", dpi=300, bbox_inches='tight')
        plt.close(fig)  # Close the figure to free memory



0: 384x640 3 Cars, 1 Motorcycle, 10 Persons, 253.1ms
Speed: 3.0ms preprocess, 253.1ms inference, 15.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 Car, 209.1ms
Speed: 16.1ms preprocess, 209.1ms inference, 11.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 5 Cars, 1 Person, 249.5ms
Speed: 0.0ms preprocess, 249.5ms inference, 18.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 4 Cars, 1 Person, 232.1ms
Speed: 0.0ms preprocess, 232.1ms inference, 15.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 Cars, 1 Motorcycle, 1 Person, 200.5ms
Speed: 0.0ms preprocess, 200.5ms inference, 16.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 Cars, 233.6ms
Speed: 0.0ms preprocess, 233.6ms inference, 20.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 4 Cars, 284.5ms
Speed: 0.0ms preprocess, 284.5ms inference, 15.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 Motorcycles, 55 Persons, 19

# Per Folder

In [66]:
import torch
import cv2
import numpy as np
from ultralytics import YOLO
import os
import supervision as sv
from PIL import Image, ImageDraw, ImageFont
from collections import Counter

# Define directory to model
model = YOLO(r"C:\Users\Spacelab3\Desktop\envs\Segmentation\Object Segmentation.v3i.yolov8\runs\segment\train6\weights\best.pt")
# Image folder directory
image_directory = "C:/Users/Spacelab3/Desktop/envs/Segmentation/Object Segmentation.v3i.yolov8/test/images"
# Output images folder
image_label = r"C:\Users\Spacelab3\Desktop\envs\Segmentation\labeled2"

# Loop through all the images in the directory
for filename in os.listdir(image_directory):
    # Check if the file is an image (you can adjust this check based on your image formats)
    if filename.endswith(('.jpg', '.jpeg', '.png')):
        image_path = os.path.join(image_directory, filename)

        # Define the path to the image directory
        image = cv2.imread(image_path)

        results = model.predict(source=image)
        detections = sv.Detections.from_ultralytics(results[0])

        numbers = detections.class_id  # Array of class IDs
        count_dict = dict(Counter(numbers))  # Count occurrences
        color_map = {
            0: ((120, 150, 170, 100), (0, 0, 0, 255)),  # Darker Light Blue
            1: ((128, 0, 128, 100), (128, 0, 128, 255)),  # Purple
            2: ((255, 0, 0, 100), (255, 0, 0, 255)),  # Red
            3: ((0, 255, 0, 100), (0, 255, 0, 255)),  # Green
            4: ((255, 20, 147, 100), (255, 20, 147, 255)) # Medium Violet Red
        }
        labels = {0: "Car", 1:"Motorcycle", 2:"Person", 3:"Truck", 4:"Van"}
        # Output the result
        print("Class Counts:", count_dict)
        # Load the image

        img = Image.fromarray(image).convert("RGBA")
        overlay = Image.new("RGBA", img.size, (255, 255, 255, 0))  # Transparent overlay
        draw_overlay = ImageDraw.Draw(overlay)

        # Iterate over each detection
        for class_id, i in zip(detections.class_id, range (len(results[0].boxes.xyxy))): 
            # Extract bounding box coordinates
            x_min, y_min, x_max, y_max = map(int, results[0].boxes.xyxy[i])  
            
            
            label = f"{labels[class_id]}"  # Use the class names
            fill_color, outline_color = color_map.get(class_id, ((0, 0, 0, 100), (0, 0, 0, 255))) # Depending on the class asign a certain color for that one

            if results[0].masks != None:                    
                mask_polygon = results[0].masks.xy[i]   
                if mask_polygon.any() != False:      
                    # Draw the polygon with transparent filling
                    draw_overlay.polygon(mask_polygon, outline=outline_color, fill=fill_color)

                # Draw the rectangle
                draw_overlay.rectangle([x_min, y_min, x_max, y_max], outline=outline_color, width=2)
                
                # Add text above the rectangle
                text_position = (x_min, y_min - 20)
                font = ImageFont.truetype("arial.ttf", 20)
                # Adjust font size or style if needed
                draw_overlay.text(text_position, label, font=font, fill=outline_color)  

        # Composite the overlay with the original image
        output_img = Image.alpha_composite(img, overlay)

        # Convert back to RGB and save or show the result
        output_img = output_img.convert("RGB")
        open_cv_image = np.array(output_img)
         # Opens the modified image
        cv2.imwrite(f"{image_label}/{filename}_labeled.jpg",open_cv_image)



0: 384x640 3 Cars, 6 Motorcycles, 3 Persons, 27.6ms
Speed: 56.0ms preprocess, 27.6ms inference, 68.8ms postprocess per image at shape (1, 3, 384, 640)
Class Counts: {2: 3, 0: 3, 1: 6}

0: 384x640 1 Car, 31.8ms
Speed: 2.5ms preprocess, 31.8ms inference, 4.2ms postprocess per image at shape (1, 3, 384, 640)
Class Counts: {0: 1}

0: 384x640 5 Cars, 26.6ms
Speed: 0.0ms preprocess, 26.6ms inference, 0.0ms postprocess per image at shape (1, 3, 384, 640)
Class Counts: {0: 5}

0: 384x640 4 Cars, 1 Person, 17.0ms
Speed: 0.0ms preprocess, 17.0ms inference, 0.0ms postprocess per image at shape (1, 3, 384, 640)
Class Counts: {0: 4, 2: 1}

0: 384x640 3 Cars, 1 Person, 19.4ms
Speed: 0.9ms preprocess, 19.4ms inference, 0.0ms postprocess per image at shape (1, 3, 384, 640)
Class Counts: {0: 3, 2: 1}

0: 384x640 6 Cars, 29.5ms
Speed: 0.0ms preprocess, 29.5ms inference, 0.0ms postprocess per image at shape (1, 3, 384, 640)
Class Counts: {0: 6}

0: 384x640 3 Cars, 11.7ms
Speed: 1.7ms preprocess, 11.7ms 

In [57]:
import cv2
import numpy as np
from PIL import Image, ImageFont
from ultralytics import YOLO
from supervision import Detections, BoxAnnotator, PolygonAnnotator
import time
# Define YOLO model path and class details
model = YOLO(r"C:\Users\Spacelab3\Desktop\envs\Segmentation\Object Segmentation.v3i.yolov8\best-150.pt")
labels = {0: "Car", 1: "Motorcycle", 2: "Person", 3: "Truck", 4: "Van"}
color_map = {
    0: (120, 150, 170),  # Light blue for Car
    1: (128, 0, 128),    # Purple for Motorcycle
    2: (255, 0, 0),      # Red for Person
    3: (0, 255, 0),      # Green for Truck
    4: (255, 20, 147),   # Pink for Van
}

# Initialize annotators
box_annotator = BoxAnnotator( thickness=2)
polygon_annotator = PolygonAnnotator( thickness=2)

# Function to calculate IoU (Intersection over Union)
def calculate_iou(box1, box2):
    """
    Calculates Intersection over Union (IoU) between two bounding boxes.

    Args:
        box1 (list or np.array): [x_min, y_min, x_max, y_max] for the first box.
        box2 (list or np.array): [x_min, y_min, x_max, y_max] for the second box.

    Returns:
        float: IoU value between the two boxes.
    """
    # Calculate the intersection coordinates
    x_min_inter = max(box1[0], box2[0])
    y_min_inter = max(box1[1], box2[1])
    x_max_inter = min(box1[2], box2[2])
    y_max_inter = min(box1[3], box2[3])

    # Calculate the area of intersection
    inter_width = max(0, x_max_inter - x_min_inter)
    inter_height = max(0, y_max_inter - y_min_inter)
    inter_area = inter_width * inter_height

    # Calculate the area of both boxes
    box1_area = (box1[2] - box1[0]) * (box1[3] - box1[1])
    box2_area = (box2[2] - box2[0]) * (box2[3] - box2[1])

    # Calculate the union area
    union_area = box1_area + box2_area - inter_area

    # Avoid division by zero
    if union_area == 0:
        return 0

    # IoU calculation
    iou = inter_area / union_area
    return iou

# Function to process YOLO predictions
def process_predictions(image_path, iou_threshold=0.3, confidence_threshold=0.3):
    """
    Processes YOLO predictions, annotates bounding boxes and masks, and checks for overlaps.

    Args:
        image_path (str): Path to the input image.
        iou_threshold (float): Threshold to flag overlapping bounding boxes.
        confidence_threshold (float): Minimum confidence score for considering a detection.

    Returns:
        Annotated image as a numpy array.
    """
    # Load the image
    image = cv2.imread(image_path)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Run YOLO predictions
    results = model.predict(source=image_rgb)
    detections = Detections.from_ultralytics(results[0])

    # Filter out detections with low confidence
    high_conf_detections = detections[detections.confidence > confidence_threshold]

    # Annotate bounding boxes for high-confidence detections
    annotated_image = box_annotator.annotate(
        scene=image_rgb,
        detections=high_conf_detections,
    )
    label_annotator = sv.LabelAnnotator(text_position=sv.Position.TOP_CENTER,text_scale=0.5)
    annotated_frame = label_annotator.annotate(
        scene=annotated_image,
        detections=detections
    )
    # Annotate segmentation masks for high-confidence detections (if available)
    if results[0].masks:
        annotated_image = polygon_annotator.annotate(scene=annotated_frame, detections=high_conf_detections)

    # Check for overlapping bounding boxes using IoU
    overlapping_boxes = []
    for i in range(len(high_conf_detections.xyxy)):
        for j in range(i + 1, len(high_conf_detections.xyxy)):
            iou = calculate_iou(high_conf_detections.xyxy[i], high_conf_detections.xyxy[j])
            if iou > iou_threshold:
                overlapping_boxes.append((i, j))

    # Log overlaps (if any)
    if overlapping_boxes:
        print(f"Overlapping bounding boxes detected: {overlapping_boxes}")
    else:
        print("No overlapping bounding boxes detected.")

    return annotated_image

# Example usage"C:\Users\Spacelab3\Desktop\envs\InstanceSegmentation\test\images\0000359_01177_d_0000705_patch_0_0_jpg.rf.1118ad7f3b95d2c9bf7cf49f1c956334.jpg"
if __name__ == "__main__":
    image_path = r'C:\Users\Spacelab3\Desktop\envs\Segmentation\Object Segmentation.v3i.yolov8\test\images\0000287_01601_d_0000767_patch_1_0_jpg.rf.5338399d71dc05541b0b86b582b3ca53.jpg' # Path to input image
    output_path = r"C:\Users\Spacelab3\Desktop\envs\Segmentation\Object Segmentation.v3i.yolov8\annotated_image.jpg"

    start_time = time.time()
    output_image = process_predictions(image_path, iou_threshold=0.3, confidence_threshold=0.3)

    # Save or display the output image
    cv2.imwrite(output_path, cv2.cvtColor(output_image, cv2.COLOR_RGB2BGR))
    total = time.time() - start_time
    print(total)
    print(f"Annotated image saved to: {output_path}")



0: 384x640 11 Cars, 5 Persons, 2 Vans, 145.3ms
Speed: 3.0ms preprocess, 145.3ms inference, 0.0ms postprocess per image at shape (1, 3, 384, 640)
Overlapping bounding boxes detected: [(14, 15), (16, 17)]
1.3762202262878418
Annotated image saved to: C:\Users\Spacelab3\Desktop\envs\Segmentation\Object Segmentation.v3i.yolov8\annotated_image.jpg


# Per image


In [62]:
from PIL import Image, ImageDraw, ImageFont
from collections import Counter
import cv2
import numpy as np
from ultralytics import YOLO
import os
import supervision as sv
import time
model = YOLO(r"C:\Users\Spacelab3\Desktop\envs\Segmentation\Object Segmentation.v3i.yolov8\runs\segment\train6\weights\best.pt")
# Define the path to the image directory
image = cv2.imread(r'C:\Users\Spacelab3\Desktop\envs\Segmentation\Object Segmentation.v3i.yolov8\test\images\0000001_05999_d_0000011_patch_0_1_jpg.rf.ef5295758cc30034bc6fa203dfd6d930.jpg')
#results = model(image) = r'C:\Users\Spacelab3\Desktop\envs\Segmentation\Object Segmentation.v3i.yolov8\test\images'
start_time = time.time()

results = model.predict(source=image)
detections = sv.Detections.from_ultralytics(results[0])

# Example detections data

numbers = detections.class_id  # Array of class IDs
count_dict = dict(Counter(numbers))  # Count occurrences

labels = {0: "Car", 1:"Motorcycle", 2:"Person", 3:"Truck"}
# Output the result
print("Class Counts:", count_dict)

# Load the image
img = Image.fromarray(image).convert("RGBA")
overlay = Image.new("RGBA", img.size, (255, 255, 255, 0))  # Transparent overlay
draw_overlay = ImageDraw.Draw(overlay)

# Iterate over each detection
for det, class_id, (i, mask_polygon) in zip(results[0].boxes.xyxy, detections.class_id, enumerate(results[0].masks.xy)):  # Assuming 'results[0].masks.xy' gives polygons
    x_min, y_min, x_max, y_max = map(int, det)  # Extract bounding box coordinates
    
     
    label = f"Class {labels[class_id]}"  # Modify this to use class names if available

    if class_id == 0:
        # Light blue
        fill_color = (173, 216, 230, 100)  # Light Blue with 100/255 alpha
        outline_color = (173, 216, 230, 255)  # Fully opaque light blue outline

    elif class_id == 1:
        # Purple
        fill_color = (128, 0, 128, 100)  # Purple with 100/255 alpha
        outline_color = (128, 0, 128, 255)  # Fully opaque purple outline        
    elif class_id == 2:
        # Red
        fill_color = (255, 0, 0, 100)  # Red with 100/255 alpha
        outline_color = (255, 0, 0, 255)  # Fully opaque red outline
    elif class_id == 3:
        # Green
        fill_color = (0, 255, 0, 100)  # Green with 100/255 alpha
        outline_color = (0, 255, 0, 255)  # Fully opaque green outline    

    # Draw the polygon with transparent filling
    #fill_color = (0, 255, 0, 100)  # Green with 100/255 alpha
    #outline_color = (0, 255, 0, 255)  # Fully opaque green outline
    draw_overlay.polygon(mask_polygon, outline=outline_color, fill=fill_color)
    # Draw the rectangle
    draw_overlay.rectangle([x_min, y_min, x_max, y_max], outline=outline_color, width=2)
    
    # Add text above the rectangle
    text_position = (x_min, y_min - 20)
    font = ImageFont.truetype("arial.ttf", 20)
    draw_overlay.text(text_position, label , font=font, fill="red")  # Adjust font size or style if needed

# Composite the overlay with the original image
output_img = Image.alpha_composite(img, overlay)

# Convert back to RGB and save or show the result
output_img = output_img.convert("RGB")  # Convert back to RGB if needed for saving
open_cv_image = np.array(output_img)

cv2.imwrite(r"C:\Users\Spacelab3\Desktop\envs\Segmentation\Object Segmentation.v3i.yolov8\annotated_image2.jpg", output_img)

total = time.time() - start_time
print(total)


0: 384x640 3 Cars, 6 Motorcycles, 3 Persons, 57.8ms
Speed: 2.1ms preprocess, 57.8ms inference, 6.0ms postprocess per image at shape (1, 3, 384, 640)
Class Counts: {2: 3, 0: 3, 1: 6}
0.8628058433532715


# Class count

In [29]:
root_dir = r"C:\Users\Spacelab3\Desktop\envs\InstanceSegmentation"  # Replace with the path to your labels directory
file_extension=".txt"
file_paths = []
class_counter = Counter()

for dirpath, dirnames, filenames in os.walk(root_dir):
    if "test" not in dirpath:
        for file in filenames:
            if file.endswith(file_extension):  # Check for the desired file extension
                full_path = os.path.join(dirpath, file)
                print(full_path)
                file_paths.append(full_path)
                with open(full_path, 'r') as f:
                            for line in f:
                                class_id = int(line.split()[0])  # Extract class ID (first value on each line)
                                class_counter[class_id] += 1                

C:\Users\Spacelab3\Desktop\envs\InstanceSegmentation\train\labels\0000001_02999_d_0000005_patch_0_0_jpg.rf.03750603bf63b592779e5dabed015e4c.txt
C:\Users\Spacelab3\Desktop\envs\InstanceSegmentation\train\labels\0000001_02999_d_0000005_patch_0_0_jpg.rf.b17b00064be2ee2d9d24a57d2c78ec2f.txt
C:\Users\Spacelab3\Desktop\envs\InstanceSegmentation\train\labels\0000001_02999_d_0000005_patch_0_0_jpg.rf.d052219754b8be902dc63a5c854a7f5e.txt
C:\Users\Spacelab3\Desktop\envs\InstanceSegmentation\train\labels\0000001_02999_d_0000005_patch_1_1_jpg.rf.0d310af5abc52703021913312f6d8d12.txt
C:\Users\Spacelab3\Desktop\envs\InstanceSegmentation\train\labels\0000001_02999_d_0000005_patch_1_1_jpg.rf.90e3149564cd8fac0ddeffc16fb691da.txt
C:\Users\Spacelab3\Desktop\envs\InstanceSegmentation\train\labels\0000001_02999_d_0000005_patch_1_1_jpg.rf.fc1efc0fefd791d3ca85edf04c7f565d.txt
C:\Users\Spacelab3\Desktop\envs\InstanceSegmentation\train\labels\0000001_03499_d_0000006_patch_0_0_jpg.rf.47f671a2bc2807309fdacb9fe0bef

In [30]:
class_counter

Counter({0: 62422, 2: 23410, 1: 7898, 3: 4281, 4: 3912})

In [ ]:
labels = {0: "Car", 1:"Motorcycle", 2:"Person", 3:"Truck", 4:"Van"}
101,923

0 - 0.3265611483130947422383134151421
1 - 2.580982527222081539630286148392
2 - 0.87076463049978641606151217428449                                    
3 - 4.7616444755898154636767110488204
4 - 5.2107873210633946830265848670757